# 12 — Measured FMM plan memory

This notebook constructs real `UniformFmm` plans and compares their recorded memory with the independent analytical model in `fmm_memory.py`. It measures three dimensions separately: particle count, Cartesian expansion order, and tree depth through level 5. This gives broad coverage without paying for a redundant full Cartesian product.

Every configuration constructs a CPU-static plan. When a working CUDA device is available, the same configuration also constructs CUDA-partial and CUDA-full plans. Both **total persistent memory** and named intermediate storage—including M2M, M2L, L2L, interaction metadata, and work buffers—are reported in decimal GB. CPU values are explicitly labelled as host memory, while CUDA values are explicitly labelled as GPU device memory.

In [ ]:
# Sweep ranges and fixed values for the other two dimensions.
PARTICLE_COUNTS = [1000, 5000, 10000, 20000, 40000]
EXPANSION_ORDERS = [2, 4, 6, 8, 10]
TREE_DEPTHS = [2, 3, 4, 5, 6]

BASE_PARTICLES = 10000
BASE_ORDER = 4
BASE_DEPTH = 3
ORDER_SWEEP_PARTICLES = 10000
CONSTRUCT_CUDA_PLANS = True
RANDOM_SEED = 314159

In [ ]:
import gc
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import cdfmm

try:
    from fmm_memory import estimate_source_point_storage
except ModuleNotFoundError:
    from examples.notebooks.fmm_memory import estimate_source_point_storage

GB = 1_000_000_000
rng = np.random.default_rng(RANDOM_SEED)
master_positions = rng.uniform(
    -0.95, 0.95, size=(max(PARTICLE_COUNTS + [BASE_PARTICLES]), 3)
)
cuda_enabled = CONSTRUCT_CUDA_PLANS and cdfmm.cuda_available()
print(f"CUDA plans will be constructed: {cuda_enabled}")
if CONSTRUCT_CUDA_PLANS and not cuda_enabled:
    print("CUDA is unavailable; all CPU measurements will still run.")


@dataclass(frozen=True)
class Configuration:
    particles: int
    order: int
    depth: int


particle_sweep = [
    Configuration(count, BASE_ORDER, BASE_DEPTH)
    for count in PARTICLE_COUNTS
]
order_sweep = [
    Configuration(ORDER_SWEEP_PARTICLES, order, BASE_DEPTH)
    for order in EXPANSION_ORDERS
]
depth_sweep = [
    Configuration(BASE_PARTICLES, BASE_ORDER, depth)
    for depth in TREE_DEPTHS
]
configurations = list(dict.fromkeys(
    particle_sweep + order_sweep + depth_sweep
))
print(f"Unique real plan configurations: {len(configurations)}")


## Measurement semantics

The estimated values are derived independently from each real tree's topology. The measured values come from plans that are actually constructed:

- CPU host-memory total: `static_plan_statistics['total_bytes']`.
- GPU device-memory total: `cuda_plan_statistics['persistent_device_bytes']`.
- Intermediate storage: operator tables, interaction metadata, scratch buffers, and individual M2M/M2L/L2L matrix tables.

The order sweep intentionally uses fewer particles because high-order M2L matrix construction depends primarily on coefficient count and translation classes, not particle count. This keeps orders through 6 interactive while still constructing genuine plans.

In [ ]:
def make_options(configuration, backend):
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = configuration.order
    options.tree.max_level = configuration.depth
    options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    options.tree.root_half_width = 1.0
    options.backend = backend
    options.fixed_target_source_indices = list(range(configuration.particles))
    return options


def construct_statistics(configuration, backend):
    positions = master_positions[:configuration.particles]
    plan = cdfmm.UniformFmm(
        positions, positions, make_options(configuration, backend)
    )
    static_statistics = dict(plan.static_plan_statistics)
    cuda_statistics = dict(plan.cuda_plan_statistics)
    del plan
    gc.collect()
    return static_statistics, cuda_statistics


measurements = {}
for index, configuration in enumerate(configurations, start=1):
    label = (
        f"N={configuration.particles}, p={configuration.order}, "
        f"depth={configuration.depth}"
    )
    print(f"[{index}/{len(configurations)}] Constructing {label}")
    positions = master_positions[:configuration.particles]
    estimate = estimate_source_point_storage(
        positions, configuration.order, configuration.depth
    )
    cpu_statistics, _ = construct_statistics(
        configuration, cdfmm.ExecutionBackend.CPU_STATIC
    )
    record = {
        "estimate": estimate,
        "cpu": cpu_statistics,
        "cuda_partial": None,
        "cuda_full": None,
    }
    if cuda_enabled:
        _, record["cuda_partial"] = construct_statistics(
            configuration, cdfmm.ExecutionBackend.CUDA_PARTIAL
        )
        _, record["cuda_full"] = construct_statistics(
            configuration, cdfmm.ExecutionBackend.CUDA_FULL
        )
    measurements[configuration] = record
    assert estimate.host_static_bytes == cpu_statistics["total_bytes"]

print("All requested plans were constructed successfully.")


In [ ]:
def total_values(configuration, backend):
    record = measurements[configuration]
    estimate = record["estimate"]
    if backend == "cpu":
        return estimate.host_static_bytes, record["cpu"]["total_bytes"]
    if backend == "cuda_partial":
        actual = record[backend]
        return (
            estimate.cuda_partial_bytes,
            None if actual is None else actual["persistent_device_bytes"],
        )
    actual = record[backend]
    return (
        estimate.cuda_full_bytes,
        None if actual is None else actual["persistent_device_bytes"],
    )


def format_gb(value):
    return "n/a" if value is None else f"{value / GB:.6f}"


print(
    f"{'N':>6s} {'Order':>5s} {'Depth':>5s} {'Memory domain':>22s} "
    f"{'Estimated GB':>14s} {'Actual GB':>12s} {'Delta GB':>12s}"
)
for configuration in configurations:
    for backend, label in [
        ("cpu", "CPU host"),
        ("cuda_partial", "GPU partial device"),
        ("cuda_full", "GPU full device"),
    ]:
        estimated, actual = total_values(configuration, backend)
        delta = None if actual is None else actual - estimated
        print(
            f"{configuration.particles:6d} {configuration.order:5d} "
            f"{configuration.depth:5d} {label:>22s} "
            f"{estimated / GB:14.6f} {format_gb(actual):>12s} "
            f"{format_gb(delta):>12s}"
        )


In [ ]:
sweeps = [
    ("Particle count", particle_sweep, lambda item: item.particles),
    ("Expansion order", order_sweep, lambda item: item.order),
    ("Tree depth", depth_sweep, lambda item: item.depth),
]
backends = [
    ("cpu", "CPU host memory"),
    ("cuda_partial", "GPU partial device memory"),
    ("cuda_full", "GPU full device memory"),
]
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
for row, (sweep_name, sweep, x_value) in enumerate(sweeps):
    x = np.array([x_value(configuration) for configuration in sweep])
    for column, (backend, backend_name) in enumerate(backends):
        axis = axes[row, column]
        values = [total_values(configuration, backend) for configuration in sweep]
        estimated = np.array([value[0] / GB for value in values])
        actual = np.array([
            np.nan if value[1] is None else value[1] / GB
            for value in values
        ])
        axis.plot(x, estimated, "o--", label="estimated")
        if np.isfinite(actual).any():
            axis.plot(x, actual, "s-", label="constructed plan")
        axis.set_title(f"{sweep_name}: {backend_name}")
        axis.set_xlabel(sweep_name)
        domain = "CPU host" if backend == "cpu" else "GPU device"
        axis.set_ylabel(f"Total persistent {domain} memory (GB)")
        axis.set_yscale("log")
        if sweep_name == "Particle count":
            axis.set_xscale("log")
        axis.grid(True, which="both", alpha=0.25)
        axis.legend()
fig.suptitle("Total FMM plan memory: CPU host versus GPU device", fontsize=15)
fig.tight_layout()
plt.show()


## Intermediate storage breakdown

The following tables preserve the total-memory view while exposing the important internal allocations. CPU aggregate categories are allocations in **host memory**. Translation matrix tables and M2L metadata are then shown individually. CUDA counters are allocations in **GPU device memory** and expose the total device allocation plus M2M, M2L, L2L, and M2L-metadata subsets; the remainder includes P2M/L2P operators, P2P tensors, geometry, coefficients, fields, and dynamic buffers. CPU host and GPU device totals are separate physical memory domains and must not be added together.

In [ ]:
representative = Configuration(BASE_PARTICLES, BASE_ORDER, max(TREE_DEPTHS))
record = measurements[representative]
estimate = record["estimate"]
cpu = record["cpu"]

print(f"Representative configuration: {representative}")
print("\nCPU HOST MEMORY: complete plan and translation intermediates")
cpu_rows = [
    ("TOTAL CPU host plan", estimate.host_static_bytes, cpu["total_bytes"]),
    ("All operators", None, cpu["operator_bytes"]),
    ("All interaction metadata", None, cpu["interaction_bytes"]),
    ("All scratch buffers", None, cpu["scratch_bytes"]),
    ("M2M matrices", estimate.host_static["shared M2M/L2L maps"] // 2, cpu["m2m_operator_bytes"]),
    ("M2L matrices", estimate.host_static["cached M2L matrices"], cpu["m2l_operator_bytes"]),
    ("M2L interaction metadata", estimate.host_static["M2L interaction indices"], cpu["m2l_interaction_bytes"]),
    ("L2L matrices", estimate.host_static["shared M2M/L2L maps"] // 2, cpu["l2l_operator_bytes"]),
]
print(f"{'CPU host component':>27s} {'Estimated GB':>14s} {'Actual GB':>12s}")
for component, estimated, actual in cpu_rows:
    print(
        f"{component:>27s} {format_gb(estimated):>14s} "
        f"{format_gb(actual):>12s}"
    )

print("\nNormalised M2L sharing")
print(f"Estimated transfer classes: {estimate.transfer_classes}")
print(f"CPU M2L matrices:          {cpu['m2l_operators']}")
print(f"Theoretical maximum:       {cpu['m2l_theoretical_maximum_classes']}")
assert cpu["m2l_operators"] == estimate.transfer_classes
assert cpu["m2l_operators"] <= cpu["m2l_theoretical_maximum_classes"]
assert cpu["m2l_operator_bytes"] == estimate.host_static["cached M2L matrices"]

for key, title, estimated_components in [
    ("cuda_partial", "CUDA partial", estimate.cuda_partial),
    ("cuda_full", "CUDA full", estimate.cuda_full),
]:
    statistics = record[key]
    print(f"\nGPU DEVICE MEMORY: {title} plan")
    if statistics is None:
        print("  unavailable in this session")
        continue
    named_actual = (
        statistics["m2m_matrix_bytes"]
        + statistics["m2l_matrix_bytes"]
        + statistics["m2l_interaction_metadata_bytes"]
        + statistics["l2l_matrix_bytes"]
    )
    print(f"  TOTAL GPU device memory: {format_gb(statistics['persistent_device_bytes'])} GB")
    print(f"  M2M matrices:            {format_gb(statistics['m2m_matrix_bytes'])} GB")
    print(f"  M2L matrices:            {format_gb(statistics['m2l_matrix_bytes'])} GB")
    print(f"  M2L metadata:            {format_gb(statistics['m2l_interaction_metadata_bytes'])} GB")
    print(f"  L2L matrices:            {format_gb(statistics['l2l_matrix_bytes'])} GB")
    print(f"  All remaining storage:   {format_gb(statistics['persistent_device_bytes'] - named_actual)} GB")
    print(f"  M2L matrix count:        {statistics['m2l_unique_matrix_count']}")
    assert statistics["m2l_unique_matrix_count"] == estimate.transfer_classes


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, (sweep_name, sweep, x_value) in zip(axes, sweeps):
    x = np.array([x_value(configuration) for configuration in sweep])
    components = {
        "M2M matrices": [measurements[item]["cpu"]["m2m_operator_bytes"] / GB for item in sweep],
        "M2L matrices": [measurements[item]["cpu"]["m2l_operator_bytes"] / GB for item in sweep],
        "M2L metadata": [measurements[item]["cpu"]["m2l_interaction_bytes"] / GB for item in sweep],
        "L2L matrices": [measurements[item]["cpu"]["l2l_operator_bytes"] / GB for item in sweep],
        "Scratch buffers": [measurements[item]["cpu"]["scratch_bytes"] / GB for item in sweep],
    }
    for component, values in components.items():
        axis.plot(x, values, marker="o", label=component)
    axis.set_title(sweep_name)
    axis.set_xlabel(sweep_name)
    axis.set_ylabel("Measured CPU host intermediate memory (GB)")
    axis.set_yscale("log")
    if sweep_name == "Particle count":
        axis.set_xscale("log")
    axis.grid(True, which="both", alpha=0.25)
    axis.legend(fontsize=8)
fig.suptitle("Measured CPU host intermediate memory across all three sweeps")
fig.tight_layout()
plt.show()


## Interpretation and limits

The CPU estimate is required to match the constructed plan's total exactly for every configuration. CUDA totals and intermediate counters are read only after a real device plan has been created. The M2L checks verify that the retained matrix count is the number of global integer transfer classes—never more than 316—and does not acquire an extra factor of tree depth or interaction count.

The reported totals are allocations owned and accounted for by the FMM plan. They exclude allocator bookkeeping, CUDA context/library allocations, Python and plotting objects, and memory owned by other processes. Thus they describe the complete persistent plan representation, while process RSS or `nvidia-smi` describes a larger runtime context.